# Stage 2 follow-up: Repartition Curated Tables
## ISM 6562 — Final Project — MediStream Telehealth
---
**One-off rewrite job.** The original `02-spark-transforms.ipynb` writes the five curated tables to Parquet without `partitionBy(...)`, so they land as monolithic files. The brief's Stage 1 guide gives explicit partitioning hints — applying them lets Spark prune entire directories on filtered reads, which materially speeds up downstream analytics and the Stage 3 streaming joins.

**Partitioning per the brief:**

| Curated table | Partition keys |
|---|---|
| `appointments`     | `specialty`, `scheduled_month` (derived from `scheduled_time`) |
| `session_quality`  | `device_type` |
| `patient_vitals`   | `measurement_type` |
| `patient_feedback` | not specified by brief — left unpartitioned |
| `physician_schedule` | not specified by brief — left unpartitioned |

**Idempotent.** Re-running this notebook is safe — it overwrites in place. After the first successful run, downstream notebooks (including the new `02b`–`02e` follow-ups) will read from the partitioned layout transparently.

## Setup

In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
    .appName('MediStream-Stage2f-RepartitionCurated')
    .master('spark://spark-master:7077')
    .config('spark.executor.memory', '1g')
    .config('spark.driver.memory', '1g')
    .config('spark.sql.shuffle.partitions', '8')
    .getOrCreate())

HDFS_CURATED  = 'hdfs://namenode:9000/medistream/curated'
LOCAL_CURATED = '/home/jovyan/data/curated'

try:
    spark.range(1).write.mode('overwrite').parquet(f'{HDFS_CURATED}/_probe')
    CURATED = HDFS_CURATED
    print('Using HDFS')
except Exception:
    CURATED = LOCAL_CURATED
    print('HDFS unavailable, using local mount')
print(f'  curated: {CURATED}')

## Helper

Read in, repartition out, write back to the same path. Uses a temp staging path so a partial-write failure can't leave the curated zone in a half-broken state.

Note: Spark's default `mode('overwrite')` is *static* — it removes the entire output directory before writing. That's what we want for a one-off layout migration.

In [ ]:
def repartition_in_place(table_name, partition_cols, transform=None):
    src = f'{CURATED}/{table_name}'
    staging = f'{CURATED}/_staging_{table_name}'
    print(f'\n--- {table_name} -> partitionBy({partition_cols}) ---')

    df = spark.read.parquet(src)
    before = df.count()
    if transform is not None:
        df = transform(df)

    # Write to staging first
    (df.write.mode('overwrite').partitionBy(*partition_cols).parquet(staging))

    # Verify staging is good before destroying source
    staged = spark.read.parquet(staging)
    after = staged.count()
    assert after == before, f'Row count mismatch: before={before:,} after={after:,}'
    print(f'  staged OK: {after:,} rows')

    # Promote staging -> source by writing again to source path then dropping staging
    (staged.write.mode('overwrite').partitionBy(*partition_cols).parquet(src))
    spark.read.parquet(src).count()  # sanity
    print(f'  promoted to {src}')

    # Clean up staging
    sc = spark.sparkContext
    hadoop_conf = sc._jsc.hadoopConfiguration()
    fs = sc._jvm.org.apache.hadoop.fs.FileSystem.get(hadoop_conf)
    staging_path = sc._jvm.org.apache.hadoop.fs.Path(staging)
    if fs.exists(staging_path):
        fs.delete(staging_path, True)
        print(f'  cleaned up {staging}')

## Repartition each table

In [ ]:
# 1. appointments — by specialty + scheduled_month (derived)
def add_scheduled_month(df):
    return df.withColumn('scheduled_month', F.date_format('scheduled_time', 'yyyy-MM'))

repartition_in_place('appointments', ['specialty', 'scheduled_month'], transform=add_scheduled_month)

In [ ]:
# 2. session_quality — by device_type
repartition_in_place('session_quality', ['device_type'])

In [ ]:
# 3. patient_vitals — by measurement_type
repartition_in_place('patient_vitals', ['measurement_type'])

## Verify the new layout

In [ ]:
# Use HDFS FileSystem API to list partition directories
sc = spark.sparkContext
hadoop_conf = sc._jsc.hadoopConfiguration()
fs = sc._jvm.org.apache.hadoop.fs.FileSystem.get(hadoop_conf)

for table in ['appointments', 'session_quality', 'patient_vitals']:
    path = sc._jvm.org.apache.hadoop.fs.Path(f'{CURATED}/{table}')
    statuses = fs.listStatus(path)
    partition_dirs = sorted([s.getPath().getName() for s in statuses if s.isDirectory()])
    print(f'\n{table}: {len(partition_dirs)} top-level partitions')
    for p in partition_dirs[:8]:
        print(f'  {p}')
    if len(partition_dirs) > 8:
        print(f'  ... and {len(partition_dirs) - 8} more')

    # Quick read-back sanity
    df = spark.read.parquet(f'{CURATED}/{table}')
    print(f'  total rows after repartition: {df.count():,}')